In [1]:
import os
os.environ["OPENAI_API_KEY"] = ""


In [12]:
import os
import json
import openai
from phi.agent import Agent
from fastmcp import Client
from dotenv import load_dotenv
import time

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")


class LLMToolAgent(Agent):
    """PhiData Agent that uses OpenAI LLM to choose MCP tools and arguments."""

    def __init__(self, client: Client, name="LLM MCP Agent"):
        super().__init__(name=name)
        self.client = client
        self.tools = {
            "add_numbers": ["a", "b"],
            "mongo_read_tool": ["filter"],
        }

    async def select_tool_and_args(self, user_input: str):
        """
        Uses OpenAI LLM to determine:
        1) Which tool to call.
        2) Extract arguments for the tool in JSON format.
        """
        prompt = f"""
You are a MongoDB query assistant. Convert natural language queries into JSON filters for the `mongo_read_tool`. 
Only output the JSON object with the "filter" key.

Collection structure:
{{
  "_id": "ObjectId",
  "type": "string",
  "serial": "string",
  "make": "string",
  "warranty_start": "date",
  "warranty_end": "date",
  "purchase_showroom": "string",
  "city": "string",
  "cost": "float",
  "shipping_cost": "float",
  "purchase_date": "date",
  "createdAt": "date",
  "updatedAt": "date",
  "__v": "int"
}}

Rules:
1. Always use valid JSON.
2. Dates must be strings in "YYYY-MM-DD" format.
3. Use MongoDB operators:
   - "$gt"  → "after" or "greater than"
   - "$lt"  → "before" or "less than"
   - "$gte" → "on or after" / "from"
   - "$lte" → "on or before" / "until"
   - For "between X and Y", use: {{"$gte": "X", "$lte": "Y"}}
4. If no date or numeric filter is given, return exact match string filters.
5. If no filters are mentioned, return an empty filter {{}}.

Examples:

User: "Show all ACs in Bangalore"
Output:
{{"filter": {{"type": "AC", "city": "Bangalore"}}}}

User: "Find Whirlpool ACs purchased after 2022-09-01"
Output:
{{"filter": {{"make": "Whirlpool", "purchase_date": {{"$gt": "2022-09-01"}}}}}}

User: "Get all appliances purchased between 2022-01-01 and 2023-01-01"
Output:
{{"filter": {{"purchase_date": {{"$gte": "2022-01-01", "$lte": "2023-01-01"}}}}}}

User: "List products whose warranty ended before 2024-01-01"
Output:
{{"filter": {{"warranty_end": {{"$lt": "2024-01-01"}}}}}}

User: "Show all Whirlpool items purchased from BestBuy in Bangalore"
Output:
{{"filter": {{"make": "Whirlpool", "purchase_showroom": "BestBuy", "city": "Bangalore"}}}}

User: "Show all documents"
Output:
{{"filter": {{}}}}

Now, given the user query below, output only the JSON filter.

User: "{user_input}"
Output:
"""
        response = openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        text_response = response.choices[0].message.content.strip()

        try:
            result = json.loads(text_response)
            print(result)
            args = result.get("filter", {})
            return "mongo_read_tool", {"filter": args}
        except json.JSONDecodeError:
            print("⚠️ Could not parse JSON from LLM:", text_response)
            return None, {}

    async def run_nlp(self, user_input: str):
        """Select tool via LLM, call MCP tool, and measure response time."""
        start_time = time.time()
        tool_name, args = await self.select_tool_and_args(user_input)

        if not tool_name:
            elapsed_time = time.time() - start_time
            return {"result": "Could not determine the tool from the input.", "time_seconds": elapsed_time}

        try:
            result_stream = await self.client.call_tool(tool_name, args)
            structured_result = getattr(result_stream, "structured_content", {})
            result = structured_result.get("result", {}).get("result")
            elapsed_time = time.time() - start_time
            return {"result": result, "time_seconds": elapsed_time}
        except Exception as e:
            elapsed_time = time.time() - start_time
            return {"result": f"Error calling tool '{tool_name}': {e}", "time_seconds": elapsed_time}


# -----------------------------
# Jupyter Notebook Usage
# -----------------------------
async def notebook_demo():
    server_url = "http://localhost:8000/mcp"

    async with Client(server_url) as mcp_client:
        await mcp_client.ping()
        print(f"Connected to MCP server at {server_url}")

        agent = LLMToolAgent(client=mcp_client)

        # Example queries to test date handling:
        test_queries = [
            #"Find Whirlpool ACs purchased after 2022-09-01"
            # "Get all products purchased between 2022-01-01 and 2023-01-01",
            # "List all appliances whose warranty ended before 2024-01-01",
            # "Show all Whirlpool items purchased on or after 2023-06-01",
            "show all appliances with purchase date after 2025-01-01 and before 2025-06-01"
        ]

        for q in test_queries:
            result = await agent.run_nlp(q)
            print(f"\nInput: {q}\nResult: {result['result']}")
            print(f"Time taken: {result['time_seconds']:.3f}s")


# In Jupyter Notebook, run:
await notebook_demo()


Connected to MCP server at http://localhost:8000/mcp
{'filter': {'purchase_date': {'$gt': '2025-01-01', '$lt': '2025-06-01'}}}

Input: show all appliances with purchase date after 2025-01-01 and before 2025-06-01
Result: []
Time taken: 7.949s
